# MoPR Geospatial Semantic Segmentation Training
## SegFormer-B2 for Indian Village Building & Infrastructure Analysis

**Hackathon:** Ministry of Panchayati Raj (MoPR) Geospatial Intelligence

**Task:** Semantic segmentation of drone orthophotos to extract:
- Building footprints by roof type (RCC, tile, tin, thatched)
- Roads (pucca/kaccha)
- Waterbodies
- Vegetation

**Model:** SegFormer-B2 with SatMAE or ImageNet pretraining

**Input:** 4-channel (RGB + normalized DSM), 512×512 tiles

**Loss:** Combined Dice + Focal loss with class weights

**Framework:** HuggingFace Transformers (lightweight, Colab-friendly)

## Cell 1: Environment Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install required packages in correct order
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers timm albumentations
!pip install -q rasterio geopandas rio-cogeo
!pip install -q wandb scikit-learn
!pip install -q tensorboard
!pip install -q tqdm numpy pandas opencv-python pillow

import torch
import torchvision
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Cell 2: Clone Repository & Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Google Drive mounted at /content/drive")

In [ ]:
# Clone the repository (update with your GitHub username)
# REPLACE: YOUR_USERNAME with your actual GitHub username
import os

repo_path = '/content/mopr-hackathon'
if os.path.exists(repo_path):
    print(f"{repo_path} already exists, skipping clone")
else:
    !git clone https://github.com/YOUR_USERNAME/mopr-hackathon.git {repo_path}

os.chdir(repo_path)
print(f"Working directory: {os.getcwd()}")
!ls -la

## Cell 3: Import Libraries & Set Random Seeds

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, SequentialLR, LinearLR
from torch.cuda.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import warnings
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, jaccard_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime
import wandb

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Libraries imported successfully")

## Cell 4: Load Configuration

In [ ]:
# Import config modules
import sys
sys.path.insert(0, '/content/mopr-hackathon')

from configs.training_config import (
    ModelConfig, DataConfig, TrainingConfig, AugmentationConfig, InferenceConfig,
    get_default_config
)
from configs.class_weights import CLASS_NAMES, CLASS_COLORS, DEFAULT_CLASS_WEIGHTS

# Get default configuration
config_dict = get_default_config()
model_cfg = config_dict['model']
data_cfg = config_dict['data']
train_cfg = config_dict['training']
aug_cfg = config_dict['augmentation']

print("Configuration loaded:")
print(f"  Model: {model_cfg.backbone}")
print(f"  Num classes: {model_cfg.num_classes}")
print(f"  In channels: {model_cfg.in_channels}")
print(f"  Batch size: {train_cfg.batch_size}")
print(f"  Learning rate: {train_cfg.lr}")
print(f"  Epochs: {train_cfg.epochs}")

## Cell 5: Initialize Weights & Biases

In [ ]:
# Initialize Weights & Biases
# You will be prompted to log in on first run
wandb.login()

# Create a run config
wandb_config = {
    'model_architecture': model_cfg.architecture,
    'backbone': model_cfg.backbone,
    'num_classes': model_cfg.num_classes,
    'in_channels': model_cfg.in_channels,
    'image_size': model_cfg.image_size,
    'batch_size': train_cfg.batch_size,
    'learning_rate': train_cfg.lr,
    'weight_decay': train_cfg.weight_decay,
    'epochs': train_cfg.epochs,
    'warmup_epochs': train_cfg.warmup_epochs,
    'fp16': train_cfg.fp16,
    'dice_weight': train_cfg.dice_weight,
    'focal_weight': train_cfg.focal_weight,
    'class_names': CLASS_NAMES,
    'class_weights': DEFAULT_CLASS_WEIGHTS.tolist(),
}

run = wandb.init(
    project="mopr-geospatial-hackathon",
    name=f"segformer-b2-{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    config=wandb_config,
    notes="Training SegFormer-B2 for Indian village semantic segmentation"
)

print(f"W&B run created: {run.name}")

## Cell 6: Define Dataset Class

In [ ]:
class SVAMITVADataset(Dataset):
    """
    Dataset for SVAMISTVA village segmentation.
    Reads 4-channel tiles (RGB + DSM) and corresponding mask tiles.
    """
    
    def __init__(
        self,
        tile_dir: str,
        mask_dir: str,
        file_list: str,
        augment: bool = False,
        aug_config: AugmentationConfig = None,
    ):
        self.tile_dir = Path(tile_dir)
        self.mask_dir = Path(mask_dir)
        self.augment = augment
        self.aug_config = aug_config or AugmentationConfig()
        
        # Read file list
        with open(file_list, 'r') as f:
            self.file_list = [line.strip() for line in f.readlines() if line.strip()]
        
        print(f"Loaded {len(self.file_list)} samples from {file_list}")
        
        # Setup augmentation pipeline
        self.transform = self._get_augmentation_pipeline()
    
    def _get_augmentation_pipeline(self):
        """Create albumentations augmentation pipeline."""
        if not self.augment:
            # Validation: only normalize
            return A.Compose([
                A.Normalize(
                    mean=self.aug_config.normalize_mean,
                    std=self.aug_config.normalize_std,
                    max_pixel_value=255.0
                ),
                ToTensorV2(),
            ], is_check_shapes=False)
        
        # Training: full augmentation pipeline
        return A.Compose([
            A.HorizontalFlip(p=self.aug_config.flip_p),
            A.VerticalFlip(p=self.aug_config.flip_p),
            A.Rotate90(p=self.aug_config.rotate90_p),
            A.Rotate(
                limit=self.aug_config.rotate_limit,
                p=self.aug_config.rotate_p,
                border_mode=cv2.BORDER_CONSTANT
            ),
            A.RandomBrightnessContrast(
                brightness_limit=self.aug_config.brightness_limit,
                contrast_limit=self.aug_config.contrast_limit,
                p=self.aug_config.brightness_contrast_p
            ),
            A.Affine(
                scale=self.aug_config.scale_range,
                p=0.2,
                mode=cv2.BORDER_CONSTANT
            ),
            A.ElasticTransform(p=self.aug_config.elastic_p, sigma=10),
            A.Normalize(
                mean=self.aug_config.normalize_mean,
                std=self.aug_config.normalize_std,
                max_pixel_value=255.0
            ),
            ToTensorV2(),
        ], is_check_shapes=False)
    
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        filename = self.file_list[idx]
        
        # Read RGB tile (3 channels)
        rgb_path = self.tile_dir / f"{filename}_rgb.tif"
        image = cv2.imread(str(rgb_path))
        if image is None:
            raise FileNotFoundError(f"RGB tile not found: {rgb_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # BGR -> RGB
        
        # Try to read DSM tile (1 channel), fallback to zeros if not found
        dsm_path = self.tile_dir / f"{filename}_dsm.tif"
        if dsm_path.exists():
            dsm = cv2.imread(str(dsm_path), cv2.IMREAD_GRAYSCALE)[:, :, np.newaxis]
            dsm = dsm.astype(np.float32)
            # Normalize DSM to roughly [-1, 1] assuming typical range
            dsm = (dsm - 128.0) / 128.0
        else:
            # Fallback: use zeros
            dsm = np.zeros((image.shape[0], image.shape[1], 1), dtype=np.float32)
        
        # Stack RGB + DSM (4 channels)
        image = np.concatenate([image, dsm], axis=2).astype(np.uint8)  # DSM as uint8 for augmentation
        
        # Read mask
        mask_path = self.mask_dir / f"{filename}_mask.tif"
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Mask not found: {mask_path}")
        mask = mask.astype(np.int64)
        
        # Apply augmentation (including normalization)
        augmented = self.transform(image=image, mask=mask)
        image = augmented['image']  # torch.Tensor, shape (4, H, W), already normalized
        mask = torch.from_numpy(augmented['mask']).long()  # torch.Tensor, shape (H, W)
        
        return {
            'image': image,
            'mask': mask,
            'filename': filename,
        }

print("SVAMITVADataset class defined")

## Cell 7: Define Loss Functions & Model Setup

In [ ]:
class DiceLoss(nn.Module):
    """Dice loss for semantic segmentation."""
    
    def __init__(self, weight=None, ignore_index=-1, smooth=1e-6):
        super().__init__()
        self.weight = weight
        self.ignore_index = ignore_index
        self.smooth = smooth
    
    def forward(self, outputs, targets):
        """
        Args:
            outputs: (B, C, H, W) logits
            targets: (B, H, W) ground truth class indices
        """
        probs = torch.softmax(outputs, dim=1)
        
        num_classes = outputs.shape[1]
        dice_loss = 0.0
        
        for c in range(num_classes):
            # Skip ignore index
            if c == self.ignore_index:
                continue
            
            pred = probs[:, c, :, :]
            target = (targets == c).float()
            
            intersection = (pred * target).sum()
            cardinality = pred.sum() + target.sum()
            
            dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
            
            if self.weight is not None:
                dice = dice * self.weight[c]
            
            dice_loss += (1.0 - dice)
        
        return dice_loss / num_classes


class FocalLoss(nn.Module):
    """Focal loss for imbalanced semantic segmentation."""
    
    def __init__(self, alpha=0.25, gamma=2.0, weight=None, ignore_index=-1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index
    
    def forward(self, outputs, targets):
        """
        Args:
            outputs: (B, C, H, W) logits
            targets: (B, H, W) ground truth class indices
        """
        # Reshape for cross entropy
        b, c, h, w = outputs.shape
        outputs = outputs.permute(0, 2, 3, 1).contiguous().view(-1, c)
        targets = targets.view(-1)
        
        # Cross entropy
        ce_loss = nn.functional.cross_entropy(
            outputs, targets,
            weight=self.weight,
            ignore_index=self.ignore_index,
            reduction='none'
        )
        
        # Compute focal loss
        p = torch.exp(-ce_loss)
        focal = self.alpha * (1 - p) ** self.gamma * ce_loss
        
        return focal.mean()


class CombinedDiceFocalLoss(nn.Module):
    """Combined Dice + Focal loss."""
    
    def __init__(
        self,
        dice_weight=0.5,
        focal_weight=0.5,
        class_weights=None,
        focal_alpha=0.25,
        focal_gamma=2.0,
        ignore_index=-1,
    ):
        super().__init__()
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight
        self.dice_loss = DiceLoss(weight=class_weights, ignore_index=ignore_index)
        self.focal_loss = FocalLoss(
            alpha=focal_alpha,
            gamma=focal_gamma,
            weight=class_weights,
            ignore_index=ignore_index,
        )
    
    def forward(self, outputs, targets):
        dice = self.dice_loss(outputs, targets)
        focal = self.focal_loss(outputs, targets)
        return self.dice_weight * dice + self.focal_weight * focal


print("Loss functions defined")

In [ ]:
# Load SegFormer model
from transformers import SegformerForSemanticSegmentation

print(f"Loading model: {model_cfg.backbone}")
model = SegformerForSemanticSegmentation.from_pretrained(
    model_cfg.backbone,
    num_labels=model_cfg.num_classes,
    ignore_mismatched_sizes=True,
)

# Adapt first convolution to accept 4 channels instead of 3
# Strategy: average the pretrained 3-channel weights and duplicate for the DSM channel
if model_cfg.in_channels == 4:
    original_conv = model.segformer.encoder.patch_embeddings.patch_embeddings.proj
    original_weights = original_conv.weight.data  # shape: (hidden_size, 3, kernel_h, kernel_w)
    
    # Create new 4-channel weights
    new_weight = torch.zeros(
        original_weights.shape[0],
        4,
        original_weights.shape[2],
        original_weights.shape[3],
        device=original_weights.device,
        dtype=original_weights.dtype,
    )
    
    # Copy RGB weights
    new_weight[:, :3, :, :] = original_weights
    # Initialize DSM channel as average of RGB
    new_weight[:, 3:, :, :] = original_weights.mean(dim=1, keepdim=True)
    
    # Replace convolution
    new_conv = nn.Conv2d(4, original_weights.shape[0], 3, padding=1, bias=original_conv.bias is not None)
    new_conv.weight = nn.Parameter(new_weight)
    if original_conv.bias is not None:
        new_conv.bias = original_conv.bias
    
    model.segformer.encoder.patch_embeddings.patch_embeddings.proj = new_conv
    print("Modified first convolution to accept 4 channels")

model = model.to('cuda')
print(f"Model loaded and moved to GPU")

## Cell 8: Create Data Loaders

In [ ]:
# Create datasets
train_dataset = SVAMITVADataset(
    tile_dir=data_cfg.tile_dir,
    mask_dir=data_cfg.mask_dir,
    file_list=data_cfg.train_list,
    augment=True,
    aug_config=aug_cfg,
)

val_dataset = SVAMITVADataset(
    tile_dir=data_cfg.tile_dir,
    mask_dir=data_cfg.mask_dir,
    file_list=data_cfg.val_list,
    augment=False,
    aug_config=aug_cfg,
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=train_cfg.batch_size,
    shuffle=True,
    num_workers=data_cfg.num_workers,
    pin_memory=data_cfg.pin_memory,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=train_cfg.batch_size * 2,
    shuffle=False,
    num_workers=data_cfg.num_workers,
    pin_memory=data_cfg.pin_memory,
    drop_last=False,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Val batches per epoch: {len(val_loader)}")

## Cell 9: Setup Training Components

In [ ]:
# Move class weights to GPU
class_weights = DEFAULT_CLASS_WEIGHTS.to('cuda')

# Loss function
criterion = CombinedDiceFocalLoss(
    dice_weight=train_cfg.dice_weight,
    focal_weight=train_cfg.focal_weight,
    class_weights=class_weights,
    focal_alpha=train_cfg.focal_alpha,
    focal_gamma=train_cfg.focal_gamma,
)
criterion = criterion.to('cuda')

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=train_cfg.lr,
    weight_decay=train_cfg.weight_decay,
)

# Learning rate scheduler
num_steps_per_epoch = len(train_loader)
total_steps = num_steps_per_epoch * train_cfg.epochs
warmup_steps = num_steps_per_epoch * train_cfg.warmup_epochs

warmup_scheduler = LinearLR(
    optimizer,
    start_factor=train_cfg.warmup_lr_init / train_cfg.lr,
    total_iters=warmup_steps,
)

cosine_scheduler = CosineAnnealingWarmRestarts(
    optimizer,
    T_0=total_steps - warmup_steps,
    T_mult=1,
    eta_min=train_cfg.lr * 0.01,
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_steps],
)

# Gradient scaler for FP16
scaler = GradScaler() if train_cfg.fp16 else None

print("Training components initialized")
print(f"  Optimizer: AdamW")
print(f"  LR scheduler: LinearWarmup + Cosine")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Total steps: {total_steps}")
print(f"  FP16: {train_cfg.fp16}")

## Cell 10: Helper Functions for Metrics

In [ ]:
def compute_metrics(predictions, targets, num_classes):
    """
    Compute per-class and macro metrics.
    
    Args:
        predictions: (N,) class predictions
        targets: (N,) ground truth classes
        num_classes: number of classes
    
    Returns:
        dict with mIoU, per-class IoU, precision, recall, f1
    """
    # Confusion matrix
    cm = confusion_matrix(targets, predictions, labels=range(num_classes))
    
    # Compute IoU per class
    iou_per_class = []
    precision_per_class = []
    recall_per_class = []
    f1_per_class = []
    
    for i in range(num_classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        
        # IoU (Jaccard)
        iou = tp / (tp + fp + fn + 1e-6)
        iou_per_class.append(iou)
        
        # Precision
        precision = tp / (tp + fp + 1e-6)
        precision_per_class.append(precision)
        
        # Recall
        recall = tp / (tp + fn + 1e-6)
        recall_per_class.append(recall)
        
        # F1
        f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
        f1_per_class.append(f1)
    
    mIoU = np.mean(iou_per_class)
    mPrecision = np.mean(precision_per_class)
    mRecall = np.mean(recall_per_class)
    mF1 = np.mean(f1_per_class)
    
    return {
        'mIoU': mIoU,
        'iou_per_class': iou_per_class,
        'precision': mPrecision,
        'precision_per_class': precision_per_class,
        'recall': mRecall,
        'recall_per_class': recall_per_class,
        'f1': mF1,
        'f1_per_class': f1_per_class,
        'confusion_matrix': cm,
    }


def plot_confusion_matrix(cm, class_names):
    """
    Plot confusion matrix and return as matplotlib figure.
    """
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Ground Truth')
    ax.set_title('Confusion Matrix')
    plt.tight_layout()
    return fig

print("Metric functions defined")

## Cell 11: Training Loop

In [ ]:
%%time

# Create checkpoint directory
checkpoint_dir = Path(train_cfg.save_dir)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

best_mIoU = 0.0
best_epoch = 0
global_step = 0

# Training loop
for epoch in range(train_cfg.epochs):
    print(f"\n{'='*80}")
    print(f"Epoch {epoch+1}/{train_cfg.epochs}")
    print(f"{'='*80}")
    
    # Training phase
    model.train()
    train_loss = 0.0
    train_loader_iter = tqdm(train_loader, desc="Training", leave=False)
    
    for batch_idx, batch in enumerate(train_loader_iter):
        images = batch['image'].to('cuda')
        masks = batch['mask'].to('cuda')
        
        optimizer.zero_grad()
        
        # Forward pass with FP16
        if train_cfg.fp16:
            with autocast():
                outputs = model(pixel_values=images)
                logits = outputs.logits  # (B, num_classes, H, W)
                loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            if train_cfg.grad_clip:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(pixel_values=images)
            logits = outputs.logits
            loss = criterion(logits, masks)
            loss.backward()
            if train_cfg.grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
            optimizer.step()
        
        scheduler.step()
        
        train_loss += loss.item()
        global_step += 1
        
        # Log every n steps
        if (batch_idx + 1) % train_cfg.log_every_n_steps == 0:
            avg_loss = train_loss / (batch_idx + 1)
            current_lr = optimizer.param_groups[0]['lr']
            wandb.log({
                'train/loss': loss.item(),
                'train/avg_loss': avg_loss,
                'learning_rate': current_lr,
                'global_step': global_step,
            })
            train_loader_iter.set_postfix({'loss': f"{avg_loss:.4f}"})
    
    avg_train_loss = train_loss / len(train_loader)
    print(f"Train Loss: {avg_train_loss:.4f}")
    
    # Validation phase
    if (epoch + 1) % train_cfg.val_every_n_epochs == 0:
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            val_loader_iter = tqdm(val_loader, desc="Validation", leave=False)
            for batch in val_loader_iter:
                images = batch['image'].to('cuda')
                masks = batch['mask'].to('cuda')
                
                outputs = model(pixel_values=images)
                logits = outputs.logits
                loss = criterion(logits, masks)
                val_loss += loss.item()
                
                # Get predictions
                preds = logits.argmax(dim=1).cpu().numpy()
                targets = masks.cpu().numpy()
                
                all_preds.append(preds.flatten())
                all_targets.append(targets.flatten())
        
        avg_val_loss = val_loss / len(val_loader)
        all_preds = np.concatenate(all_preds)
        all_targets = np.concatenate(all_targets)
        
        # Compute metrics
        metrics = compute_metrics(all_preds, all_targets, model_cfg.num_classes)
        mIoU = metrics['mIoU']
        
        print(f"Val Loss: {avg_val_loss:.4f}")
        print(f"mIoU: {mIoU:.4f}")
        print(f"mPrecision: {metrics['precision']:.4f}")
        print(f"mRecall: {metrics['recall']:.4f}")
        print(f"mF1: {metrics['f1']:.4f}")
        
        # Per-class metrics
        print("\nPer-class IoU:")
        for i, (class_name, iou) in enumerate(zip(CLASS_NAMES, metrics['iou_per_class'])):
            print(f"  {class_name:<20s}: {iou:.4f}")
        
        # Log to W&B
        wandb.log({
            'val/loss': avg_val_loss,
            'val/mIoU': mIoU,
            'val/precision': metrics['precision'],
            'val/recall': metrics['recall'],
            'val/f1': metrics['f1'],
            'epoch': epoch + 1,
        })
        
        # Per-class metrics to W&B
        for i, (class_name, iou) in enumerate(zip(CLASS_NAMES, metrics['iou_per_class'])):
            wandb.log({f"val/iou_{class_name}": iou})
        
        # Log confusion matrix every 10 epochs
        if (epoch + 1) % 10 == 0:
            cm_fig = plot_confusion_matrix(metrics['confusion_matrix'], CLASS_NAMES)
            wandb.log({"val/confusion_matrix": wandb.Image(cm_fig)})
            plt.close(cm_fig)
        
        # Save best model
        if mIoU > best_mIoU:
            best_mIoU = mIoU
            best_epoch = epoch + 1
            checkpoint_path = checkpoint_dir / f"best_model.pt"
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'mIoU': mIoU,
                'config': {
                    'model': model_cfg.__dict__,
                    'data': data_cfg.__dict__,
                    'training': train_cfg.__dict__,
                },
            }, checkpoint_path)
            print(f"\n>>> Saved best model to {checkpoint_path}")
        
        # Save periodic checkpoints
        if (epoch + 1) % train_cfg.save_every_n_epochs == 0:
            checkpoint_path = checkpoint_dir / f"checkpoint_epoch_{epoch+1:03d}.pt"
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'mIoU': mIoU,
            }, checkpoint_path)
            print(f"Saved checkpoint to {checkpoint_path}")

print(f"\nTraining completed!")
print(f"Best mIoU: {best_mIoU:.4f} (Epoch {best_epoch})")

## Cell 12: Final Validation & Per-Class Metrics Table

In [ ]:
# Load best model
best_model_path = checkpoint_dir / "best_model.pt"
checkpoint = torch.load(best_model_path, map_location='cuda')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from {best_model_path}")

# Final validation
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Final validation"):
        images = batch['image'].to('cuda')
        masks = batch['mask'].to('cuda')
        
        outputs = model(pixel_values=images)
        logits = outputs.logits
        
        preds = logits.argmax(dim=1).cpu().numpy()
        targets = masks.cpu().numpy()
        
        all_preds.append(preds.flatten())
        all_targets.append(targets.flatten())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

# Final metrics
final_metrics = compute_metrics(all_preds, all_targets, model_cfg.num_classes)

print("\n" + "="*80)
print("FINAL VALIDATION METRICS")
print("="*80)
print(f"mIoU:       {final_metrics['mIoU']:.4f}")
print(f"Precision:  {final_metrics['precision']:.4f}")
print(f"Recall:     {final_metrics['recall']:.4f}")
print(f"F1 Score:   {final_metrics['f1']:.4f}")
print()

# Per-class table
import pandas as pd

metrics_table = pd.DataFrame({
    'Class': CLASS_NAMES,
    'IoU': [f"{x:.4f}" for x in final_metrics['iou_per_class']],
    'Precision': [f"{x:.4f}" for x in final_metrics['precision_per_class']],
    'Recall': [f"{x:.4f}" for x in final_metrics['recall_per_class']],
    'F1-Score': [f"{x:.4f}" for x in final_metrics['f1_per_class']],
})

print("Per-Class Metrics:")
print(metrics_table.to_string(index=False))

# Save table
table_path = checkpoint_dir / "metrics_table.csv"
metrics_table.to_csv(table_path, index=False)
print(f"\nSaved metrics table to {table_path}")

## Cell 13: Save Final Checkpoint & Push to W&B

In [ ]:
# Save final checkpoint with config
final_checkpoint_path = checkpoint_dir / "final_model.pt"
torch.save({
    'epoch': train_cfg.epochs,
    'model_state_dict': model.state_dict(),
    'mIoU': final_metrics['mIoU'],
    'config': {
        'model': {k: v for k, v in model_cfg.__dict__.items() if not k.startswith('_')},
        'data': {k: v for k, v in data_cfg.__dict__.items() if not k.startswith('_')},
        'training': {k: v for k, v in train_cfg.__dict__.items() if not k.startswith('_')},
    },
    'metrics': {
        'mIoU': final_metrics['mIoU'],
        'precision': final_metrics['precision'],
        'recall': final_metrics['recall'],
        'f1': final_metrics['f1'],
    },
}, final_checkpoint_path)

print(f"Saved final model to {final_checkpoint_path}")

# Log final metrics to W&B
wandb.log({
    'final/mIoU': final_metrics['mIoU'],
    'final/precision': final_metrics['precision'],
    'final/recall': final_metrics['recall'],
    'final/f1': final_metrics['f1'],
})

# Upload model artifact to W&B
model_artifact = wandb.Artifact('segformer-b2-model', type='model')
model_artifact.add_file(str(final_checkpoint_path))
wandb.log_artifact(model_artifact)

print(f"\nUploaded model artifact to W&B")
print(f"Run: {wandb.run.name}")
print(f"URL: {wandb.run.url}")

wandb.finish()
print("\nTraining complete!")